# The `.td` accessors on pandas and xarray

`import tensordir` registers a `.td` accessor on `pandas.DataFrame`, `pandas.Series`,
`xarray.DataArray` and `xarray.Dataset`. It offers `where` with the condition grammar of
`td.where`, and `diff` on table rows. This notebook filters a response table and a labeled
response array with them, and shows how to skip the accessors (and the pandas and xarray
imports they need) with `TENSORDIR_NO_ACCESSORS=1`.

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

import tensordir as td

root = Path(tempfile.mkdtemp(prefix="tensordir-accessors-"))
td.set_root(root)

## Responses to filter

A toy set of responses of motion-sensitive cell types to moving edges, as a labeled
`DataArray` (flyvis's `.custom.where` worked on such arrays) and as a long table.

In [ ]:
rng = np.random.default_rng(0)
cell_types = ["T4a", "T4b", "T4c", "T5a", "T5b", "T5c"]
angles = np.arange(0, 360, 45)
time = np.linspace(-0.5, 1.5, 41)
responses = xr.DataArray(
    rng.normal(size=(len(cell_types), 2, len(angles), len(time))).cumsum(-1) * 0.1,
    dims=("cell", "intensity", "angle", "time"),
    coords={"cell_type": ("cell", cell_types), "intensity": [0, 1], "angle": angles, "time": time},
    name="voltage",
)
peaks = responses.sel(time=slice(0, 1)).max("time").to_dataframe().reset_index()
peaks.head()

## `where` on xarray: coordinates

Conditions name coordinates; a list means "any of", comparison strings are combined with AND.

In [ ]:
on_edges = responses.td.where(cell_type=["T4a", "T4b"], intensity=1, time=">0, <1")
print(dict(on_edges.sizes), on_edges.cell_type.values)

In [ ]:
dataset = responses.to_dataset().assign(baseline=responses.sel(time=slice(None, 0)).mean("time"))
dataset.td.where(cell_type=td.any_of("T5a", "T5c"), angle=">=180")

## `where` on DataFrames: columns and index levels

A condition name matches a column, else `spec.<name>` (for tables of records), else an index
level.

In [ ]:
peaks.td.where(cell_type=["T4a", "T5a"], angle=td.any_of(0, 180), voltage=">0.2")

In [ ]:
by_type = peaks.set_index(["cell_type", "intensity", "angle"])
by_type.td.where(cell_type="T4c", intensity=0, angle="<=90")

## `where` on a Series: values and index levels

A positional condition filters the values; keyword conditions filter index levels.

In [ ]:
voltage = by_type["voltage"]
voltage.td.where(">0.3", cell_type=["T4a", "T4b"], intensity=1)

## `diff` on tables of records

`table.td.diff(a, b)` compares two rows (index labels) on their `spec.` columns, and
`row.td.diff(other)` compares a row with another row or a mapping.

In [ ]:
class Fit(td.Record):
    def __init__(self, /, *, cell_type: str = "T4a", lr: float = 0.1, seed: int = 0):
        self.score = {"r": float(np.random.default_rng(seed).uniform())}


for cell_type in ("T4a", "T5a"):
    for seed in (0, 1):
        Fit(cell_type=cell_type, seed=seed)
Fit(lr=0.01)

table = Fit.info(sizes=False)
print(table.td.diff(0, 1))
print(table.iloc[0].td.diff({"cell_type": "T4a", "lr": 0.1, "seed": 3}))
table.td.where(cell_type="T4a", lr=">0.05")[["key", "spec.cell_type", "spec.lr", "spec.seed"]]

## Without accessors

Registering the accessors imports pandas and xarray, which costs about 0.45 s at
`import tensordir`. Setting `TENSORDIR_NO_ACCESSORS=1` before the import skips both the
registration and the imports. The functions do the same filtering without accessors:
`td.where(obj, ...)`, or `obj.pipe(td.where, ...)` in a method chain.

In [ ]:
probe = """
import sys, time
t0 = time.perf_counter()
import tensordir
elapsed = time.perf_counter() - t0
import pandas as pd
print(f"{elapsed:.2f} s", hasattr(pd.DataFrame, "td"))
"""


def fresh_import(**env):
    result = subprocess.run([sys.executable, "-c", probe], env={**os.environ, **env}, capture_output=True, text=True)
    return result.stdout.strip()


fresh_import()  # warm the file system cache
print("with accessors:    ", fresh_import())
print("without accessors: ", fresh_import(TENSORDIR_NO_ACCESSORS="1"))

In [ ]:
same = td.where(responses, cell_type=["T4a", "T4b"], intensity=1, time=">0, <1")
print(same.identical(on_edges), peaks.pipe(td.where, cell_type="T4a", voltage=">0.2").shape)

If another package registered `.td` first, tensordir warns and keeps that accessor; the
functions work either way.